# Current human annotation summary

Reads the four current CSVs, verifies IDs and answer text against saved predictions,
excludes wholly empty rows, and refreshes `accuracy_summary.json` with source hashes.
The author-confirmed rubric is binary completeness: no missing reference details.
Claude ID 14 and Gemma IDs 14/99 have logged corrections. `Strict` remains a
diagnostic column with no separately confirmed scoring instructions.
Individual evaluator ratings are unavailable; this notebook cannot compute
inter-annotator agreement.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

root = Path.cwd().resolve()
while not (root / "scripts" / "paper_analysis.py").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook inside the LEGAL-UQA checkout")
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from scripts.paper_analysis import summarize_human_scores, annotation_frame
summary = summarize_human_scores()
display(pd.DataFrame(summary["models"]))
strict_rows = []
for model in summary["models"]:
    frame = annotation_frame(root / model["source_file"])
    strict_rows.append({"model": model["model"],
        "Strict mean (%)": 100 * frame["Strict"].mean(),
        "Strict full": int(frame["Strict"].eq(1).sum()),
        "Strict half": int(frame["Strict"].eq(.5).sum()),
        "Strict zero": int(frame["Strict"].eq(0).sum())})
display(pd.DataFrame(strict_rows))
